In [10]:
import json
import re
import os
import logging

# ================= 配置区域 =================
INPUT_DIR = "data/raw_data/十万个梗库/"
OUTPUT_FILENAME = "data/input_data/meme_rag_ready.jsonl"

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

def parse_txt_content_universal(text):
    """
    V6 解析逻辑：抛弃数字序号，只认【关键词】锚点
    """
    data = {}
    
    # 1. 预处理：先把干扰正则的 Markdown 加粗符号(**)全部删掉
    # 这样 `**【梗名称】**` 就会变成干净的 `【梗名称】`
    clean_text = text.replace("**", "")
    
    # 2. 定义正则：只找锚点
    # (?s) 开启多行模式
    # 逻辑：找到 【xxx】，忽略中间的冒号/空格，提取内容，直到遇到下一个 【xxx】 或文件结束
    
    # (1) 提取梗名
    # 匹配：【梗名称】或【名称】 -> 内容 -> 直到遇到 【梗来源】或【来源】
    meme_match = re.search(r"【(?:梗名称|名称)】[:：]?\s*(.*?)\s*(?=【(?:梗来源|来源)|$)", clean_text, re.DOTALL)
    data['meme'] = meme_match.group(1).strip() if meme_match else "暂无"

    # (2) 提取来源
    # 匹配：【梗来源】或【来源】 -> 内容 -> 直到遇到 【梗含义】或【含义】
    origin_match = re.search(r"【(?:梗来源|来源)】[:：]?\s*(.*?)\s*(?=【(?:梗含义|含义)|$)", clean_text, re.DOTALL)
    data['origin'] = origin_match.group(1).strip() if origin_match else "暂无"

    # (3) 提取含义
    # 匹配：【梗含义】或【含义】 -> 内容 -> 直到遇到 【用法建议】或【用法】
    meaning_match = re.search(r"【(?:梗含义|含义)】[:：]?\s*(.*?)\s*(?=【(?:用法建议|用法)|$)", clean_text, re.DOTALL)
    data['meaning'] = meaning_match.group(1).strip() if meaning_match else "暂无"

    # (4) 提取用法
    # 匹配：【用法建议】或【用法】 -> 内容 -> 直到文件结束
    usage_match = re.search(r"【(?:用法建议|用法)】[:：]?\s*(.*?)\s*$", clean_text, re.DOTALL)
    data['usage'] = usage_match.group(1).strip() if usage_match else "暂无"

    return data

def main():
    try:
        # 0. 准备输出目录
        output_dir = os.path.dirname(OUTPUT_FILENAME)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir, exist_ok=True)

        print(f"🚀 开始 V6 强力扫描: {INPUT_DIR}")
        
        processed_docs = []
        file_count = 0
        success_count = 0
        
        # 遍历所有文件
        for root, dirs, files in os.walk(INPUT_DIR):
            for file in files:
                if file.endswith(".txt"):
                    file_count += 1
                    file_path = os.path.join(root, file)

                    try:
                        with open(file_path, 'r', encoding='utf-8') as f:
                            content = f.read()
                        
                        parsed_data = parse_txt_content_universal(content)
                        
                        # 只要有梗名或含义就算成功
                        if parsed_data['meme'] != "暂无" or parsed_data['meaning'] != "暂无":
                            
                            # 兜底：如果没提取到梗名，用文件名
                            if parsed_data['meme'] == "暂无":
                                # 去掉 .txt, _1, 以及文件名中可能的 #标签
                                clean_name = file.replace(".txt", "").replace("_1", "").split("#")[0].strip()
                                parsed_data['meme'] = clean_name

                            doc = {
                                "page_content": (
                                    f"网络热梗：{parsed_data['meme']}\n"
                                    f"含义解析：{parsed_data['meaning']}\n"
                                    f"起源/出处：{parsed_data['origin']}\n"
                                    f"用法建议/例句：\n{parsed_data['usage']}"
                                ),
                                "metadata": {
                                    "meme_name": parsed_data['meme'],
                                    "source": "xhs_100k_memes",
                                    "original_file": file_path,
                                    "is_offensive": False
                                }
                            }
                            processed_docs.append(doc)
                            success_count += 1
                        
                        if file_count % 500 == 0:
                            print(f"⚡ 已扫描 {file_count} 个，成功 {success_count} 个...", end='\r')

                    except Exception as e:
                        pass # 跳过坏文件

        print("\n" + "=" * 40)
        print(f"📊 V6 最终战报:")
        print(f"   - 扫描总数: {file_count}")
        print(f"   - ✅ 成功转换: {success_count}")
        print(f"   - 成功率: {success_count/file_count*100:.1f}%")

        if success_count > 0:
            with open(OUTPUT_FILENAME, 'w', encoding='utf-8') as f:
                for doc in processed_docs:
                    f.write(json.dumps(doc, ensure_ascii=False) + '\n')
            print(f"✅ 文件已生成: {OUTPUT_FILENAME}")
    
    except Exception as e:
        print(f"💥 脚本崩溃: {e}")

if __name__ == "__main__":
    main()

🚀 开始 V6 强力扫描: data/raw_data/十万个梗库/
⚡ 已扫描 500 个，成功 500 个...
📊 V6 最终战报:
   - 扫描总数: 517
   - ✅ 成功转换: 517
   - 成功率: 100.0%
✅ 文件已生成: data/input_data/meme_rag_ready.jsonl
